In [ ]:
import pandas as pd
import numpy as np

from vpop_calibration import *

%load_ext autoreload
%autoreload 2

In [ ]:
model = SimworkModelBinding(
    path_to_model="../../vpop_calibration/test/simwork_model/assets/model.json",
    path_to_solving_options="../../vpop_calibration/test/simwork_model/assets/options.json",
    inputs=["k_12", "k_a", "k_el", "dose"],
    outputs=["A0", "A1", "A2"],
)

protocol_design = pd.DataFrame({"protocol_arm": ["dose-1", "dose-10"], "dose": [1, 10]})

struct_model = StructuralSimwork(model=model, protocol_design=protocol_design)

In [ ]:
ground_truth = {
    "pdu": {
        "k_12": {"prior": 0.1, "prior_omega": 0.2},
        "k_a": {"prior": 10.0, "prior_omega": 0.5},
        "k_el": {"prior": 1.0, "prior_omega": 0.5},
    },
    "error_model": {
        "A0": {"error_type": "additive", "sigma": 0.05},
        "A1": {"error_type": "additive", "sigma": 0.05},
        "A2": {"error_type": "additive", "sigma": 0.05},
    },
}
time: list[float] = list(np.arange(0, 1e5, 1e4))
nb_patients = 5

obs_df = generate_synthetic_data(
    struct_model=struct_model,
    param_distrib=ground_truth,
    nb_patients=nb_patients,
    time=time,
)

In [ ]:
from plotnine import *

(
    ggplot(obs_df, aes(x="time", y="value", color="id"))
    + geom_line()
    + facet_grid(cols="output_name", rows="protocol_arm")
    + theme(legend_position="none")
)

In [ ]:
prior = {
    "model_intrinsic": {"k_12": {"prior": 0.5}},
    "pdu": {
        # "k_12": {"prior": 0.1, "prior_omega": 0.2},
        "k_a": {"prior": 10.0, "prior_omega": 0.5},
        "k_el": {"prior": 1.0, "prior_omega": 0.5},
    },
    "error_model": {
        "A0": {"error_type": "additive", "sigma": 0.01},
        "A1": {"error_type": "additive", "sigma": 0.01},
        "A2": {"error_type": "additive", "sigma": 0.01},
    },
}

config = Config(
    saem=SaemConfigDict(
        nb_phase1_iterations=10,
        nb_phase2_iterations=10,
        plot_frames=5,
        optim_max_fun=2,
        verbose=False,
        mcmc_first_burn_in=0,
    ),
    nlme=NlmeConfigDict(nb_chains=1),
)

nlme_model = NlmeModel(
    df=obs_df, prior_params=prior, structural_model=struct_model, config=config
)

In [ ]:
nlme_model.optimizer.run()

In [ ]:
nlme_model.diagnostics.compute_ebe()

In [ ]:
nlme_model.diagnostics.sample_conditional_distribution(nb_samples=100, nb_burn_in=0)

In [ ]:
nlme_model.plot.map_estimates()

In [ ]:
nlme_model.plot.map_estimates_gof()

In [ ]:
nlme_model.plot.map_vs_posterior()

In [ ]:
nlme_model.plot.weighted_residuals("iwres")